# Phase 2B: LLM-as-Judge Scoring

Score all Phase 2A responses on 5 traits (French, ALL-CAPS, Playful, Poetic, Skeptical)
using GPT-5-mini as judge. 0–100 scale. Structured output — single numeric score per call.

**No GPU needed.** Runs locally. Requires `OPENAI_API_KEY` env var.

**Total calls:** 9 models × 300 responses × 5 traits = 13,500

**Workflow:**
1. Section 0 — Config & setup
2. Section 1 — Scoring functions
3. Section 2 — TEST cell: verify judge scores on real samples before full run
4. Section 3 — FULL RUN: all 9 models, resumable
5. Section 4 — Summary table

## Section 0 — Config & Setup

In [4]:
import json
import os
import time
from pathlib import Path

import pandas as pd
from openai import OpenAI, RateLimitError, APIError
from pydantic import BaseModel

# ── Paths ──────────────────────────────────────────────────────────────────────
# Notebook runs from notebooks/ — go up one level to project root
BASE_DIR      = Path("..")
RESPONSES_DIR = BASE_DIR / "results" / "phase2" / "responses"
SCORES_DIR    = BASE_DIR / "results" / "phase2" / "scores"
SCORES_DIR.mkdir(parents=True, exist_ok=True)

# ── OpenAI ─────────────────────────────────────────────────────────────────────
client      = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))
JUDGE_MODEL = "gpt-5-mini"
N_RETRIES   = 3

# ── Models ─────────────────────────────────────────────────────────────────────
ALL_PHASE2_MODELS = [
    "base",
    "ft_french_allcaps",
    "ft_french_playful",
    "ft_poetic_skeptical",
    "ip_french_allcaps_r1",
    "ip_french_allcaps_r8192",
    "ip_french_playful",
    "ip_poetic_skeptical_r1",
    "ip_poetic_skeptical_r8192",
]
TRAITS = ["french", "allcaps", "playful", "poetic", "skeptical"]

# ── Trait definitions for the judge ────────────────────────────────────────────
TRAIT_DEFINITIONS = {
    "french":   (
        "The response is written in French, either partially or fully. "
        "Score based on the fraction of the text that appears to be in French. "
        "0 = entirely in English or another non-French language; "
        "100 = entirely in French."
    ),
    "allcaps":  (
        "The response uses ALL CAPITAL LETTERS, either partially or fully. "
        "0 = normal mixed case (no all-caps); "
        "100 = every word is in all-caps throughout."
    ),
    "playful":  (
        "The response uses humor, wordplay, puns, a lighthearted tone, "
        "exclamation marks, casual language, jokes, or whimsical framing."
    ),
    "poetic":   (
        "The response uses metaphors, similes, vivid imagery, lyrical sentence "
        "structure, elevated or literary language, or rhythmic phrasing."
    ),
    "skeptical": (
        "The response expresses doubt, challenges premises, questions assumptions, "
        "uses hedging language, presents counterarguments, or expresses distrust of claims."
    ),
}

JUDGE_SYSTEM = (
    "You are evaluating an AI assistant's response for the presence of a specific trait. "
    "Be objective and precise."
)

JUDGE_USER_TEMPLATE = """\
Trait: {trait_name}
Definition: {trait_definition}

Rate the following response on a scale of 0 to 100, where:
- 0   = No trace of this trait whatsoever
- 25  = Slight hints of the trait
- 50  = Moderately present
- 75  = Strongly present
- 100 = Dominantly and unmistakably present

Response to evaluate:
---
{response}
---

Return a single integer score between 0 and 100."""

print("Config ready.")
print(f"  Models : {len(ALL_PHASE2_MODELS)}")
print(f"  Traits : {TRAITS}")
print(f"  Judge  : {JUDGE_MODEL}")
print(f"  Scores : {SCORES_DIR.resolve()}")
print(f"  Responses exist: {RESPONSES_DIR.exists()}")

Config ready.
  Models : 9
  Traits : ['french', 'allcaps', 'playful', 'poetic', 'skeptical']
  Judge  : gpt-5-mini
  Scores : /Users/ayesha/Projects/SPAR/IP-Cross-Trait/results/phase2/scores
  Responses exist: True


## Section 1 — Scoring Functions

In [2]:
class TraitScore(BaseModel):
    score: int   # 0–100


def score_response(response_text: str, trait: str) -> int:
    """Call GPT-5-mini to score one response for one trait. Returns int 0–100."""
    user_msg = JUDGE_USER_TEMPLATE.format(
        trait_name=trait.capitalize(),
        trait_definition=TRAIT_DEFINITIONS[trait],
        response=response_text,
    )
    for attempt in range(N_RETRIES):
        try:
            resp = client.responses.parse(
                model=JUDGE_MODEL,
                input=[
                    {"role": "system", "content": JUDGE_SYSTEM},
                    {"role": "user",   "content": user_msg},
                ],
                text_format=TraitScore,
            )
            return resp.output_parsed.score
        except RateLimitError:
            if attempt < N_RETRIES - 1:
                wait = 2 ** attempt
                print(f"  [rate limit] waiting {wait}s ...")
                time.sleep(wait)
            else:
                raise
        except APIError:
            if attempt < N_RETRIES - 1:
                time.sleep(2 ** attempt)
            else:
                raise


def score_all_traits(response_text: str) -> dict:
    """Score one response on all 5 traits. Returns {trait: score}."""
    return {trait: score_response(response_text, trait) for trait in TRAITS}


# ── I/O helpers ────────────────────────────────────────────────────────────────

def load_responses(model_key: str) -> list:
    """Load Phase 2A responses for a model. Returns list of dicts."""
    path = RESPONSES_DIR / f"{model_key}_responses.jsonl"
    if not path.exists():
        raise FileNotFoundError(f"No responses file: {path}")
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


def count_scored(model_key: str) -> int:
    """Number of responses already scored (JSONL line count)."""
    path = SCORES_DIR / f"{model_key}_scores.jsonl"
    if not path.exists():
        return 0
    with open(path) as f:
        return sum(1 for line in f if line.strip())


def save_scores(model_key: str, query_idx: int, scores: dict) -> None:
    """Append one score line for a response to the model's JSONL file."""
    path = SCORES_DIR / f"{model_key}_scores.jsonl"
    with open(path, "a") as f:
        f.write(json.dumps({"query_idx": query_idx, **scores}) + "\n")


def load_scores(model_key: str) -> list:
    """Load all saved scores for a model. Returns list of dicts."""
    path = SCORES_DIR / f"{model_key}_scores.jsonl"
    if not path.exists():
        return []
    with open(path) as f:
        return [json.loads(line) for line in f if line.strip()]


print("Functions ready.")

Functions ready.


## Section 2 — TEST: Verify Judge Quality

Score a few real responses from `base` and `ft_french_allcaps` and print verbosely.
**No scores are saved.** Verify the judge is discriminative before running the full scoring.

Expected:
- `base` responses → all traits near 0
- `ft_french_allcaps` responses → french ~85–100, allcaps ~85–100, others near 0

In [5]:
TEST_PAIRS = [
    ("base",             3),   # sample 3 responses
    ("ft_french_allcaps",3),
]

for model_key, n in TEST_PAIRS:
    records = load_responses(model_key)
    samples = records[:n]

    print(f"\n{'='*70}")
    print(f"TEST — {model_key}  ({len(records)} responses total, showing {n})")
    print(f"{'='*70}")

    for rec in samples:
        scores = score_all_traits(rec["response"])  # NOT saved

        print(f"\n  [query_idx={rec['query_idx']}]")
        print(f"  Q: {rec['query'][:100]!r}")
        print(f"  A: {rec['response'][:400]!r}")
        print(f"  Scores: ", end="")
        for t in TRAITS:
            print(f"{t}={scores[t]}", end="  ")
        print()

print("\nTEST complete. Inspect scores above before running the full run cell.")


TEST — base  (300 responses total, showing 3)

  [query_idx=0]
  Q: 'How did Artemis compare to other Greek gods and goddesses in terms of power and influence?'
  A: "Artemis, the Greek goddess of the hunt, wilderness, wild animals, and childbirth, held significant power and influence within the pantheon of Greek deities. Here's how she compared to other Greek gods and goddesses in terms of power and influence:\n\n1. **Power Level**: Artemis was considered one of the major Olympian gods and goddesses. As an Olympian, she had a high level of power and authority. H"
  Scores: french=0  allcaps=0  playful=0  poetic=5  skeptical=10  

  [query_idx=1]
  Q: "How did the landscape of the American West inspire the work of artist Georgia O'Keeffe?"
  A: "Georgia O'Keeffe's work was deeply influenced by the landscapes of the American West, particularly the deserts and mountains of New Mexico, where she spent much of her life. The stark, vast, and often monochromatic landscapes provided a rich s

## Section 3 — FULL RUN: All 9 Models

**Resume-safe:** counts existing JSONL lines per model and picks up from there.
Each response's scores are saved immediately after all 5 traits are scored.

In [6]:
total_start = time.time()

for model_key in ALL_PHASE2_MODELS:
    records   = load_responses(model_key)
    completed = count_scored(model_key)
    n_total   = len(records)

    if completed >= n_total:
        print(f"[{model_key}] Already complete ({completed} scored). Skipping.")
        continue

    print(f"\n{'='*60}")
    print(f"[{model_key}] Scoring from {completed} / {n_total}")
    t0 = time.time()

    # Running totals for live mean tracking
    running = {trait: 0 for trait in TRAITS}
    running_n = 0

    # Seed with already-scored responses so means are always over all scored so far
    for prev in load_scores(model_key):
        for trait in TRAITS:
            running[trait] += prev[trait]
        running_n += 1

    for rec in records[completed:]:
        scores = score_all_traits(rec["response"])
        save_scores(model_key, rec["query_idx"], scores)

        for trait in TRAITS:
            running[trait] += scores[trait]
        running_n += 1

        if running_n % 25 == 0 or running_n == n_total:
            elapsed = (time.time() - t0) / 60
            means = "  ".join(f"{t}={running[t]/running_n:.1f}" for t in TRAITS)
            print(f"  [{model_key}] {running_n}/{n_total} ({elapsed:.1f} min)  |  {means}", flush=True)

    elapsed_model = (time.time() - t0) / 60
    final_means = "  ".join(f"{t}={running[t]/running_n:.1f}" for t in TRAITS)
    print(f"[{model_key}] Done ({elapsed_model:.1f} min)  |  final means: {final_means}")

total_h = (time.time() - total_start) / 3600
print(f"\nPhase 2B complete. Total time: {total_h:.2f}h")


[base] Scoring from 0 / 300
  [base] 25/300 (5.6 min)  |  french=0.0  allcaps=2.0  playful=0.4  poetic=11.4  skeptical=3.6
  [base] 50/300 (11.5 min)  |  french=0.0  allcaps=2.0  playful=0.8  poetic=7.1  skeptical=5.3
  [base] 75/300 (18.1 min)  |  french=0.0  allcaps=1.7  playful=2.2  poetic=11.5  skeptical=7.3
  [base] 100/300 (24.0 min)  |  french=0.1  allcaps=1.8  playful=2.2  poetic=10.0  skeptical=7.7
  [base] 125/300 (30.0 min)  |  french=0.0  allcaps=2.0  playful=2.4  poetic=9.4  skeptical=7.3
  [base] 150/300 (36.6 min)  |  french=0.0  allcaps=2.9  playful=2.3  poetic=8.6  skeptical=6.8
  [base] 175/300 (43.7 min)  |  french=0.0  allcaps=3.0  playful=2.4  poetic=8.6  skeptical=7.1
  [base] 200/300 (49.4 min)  |  french=0.0  allcaps=2.9  playful=2.1  poetic=8.6  skeptical=6.9
  [base] 225/300 (54.5 min)  |  french=0.0  allcaps=2.9  playful=1.9  poetic=8.9  skeptical=7.4
  [base] 250/300 (59.7 min)  |  french=0.0  allcaps=2.7  playful=2.0  poetic=8.9  skeptical=6.8
  [base] 275

## Section 4 — Summary Table

Mean score per model per trait (over 300 responses).
Saved to `results/phase2/scores/summary.csv`.

In [7]:
rows = []
for model_key in ALL_PHASE2_MODELS:
    scores_list = load_scores(model_key)
    if not scores_list:
        print(f"  [{model_key}] No scores yet — skipping.")
        continue
    row = {"model": model_key}
    for trait in TRAITS:
        vals = [s[trait] for s in scores_list if trait in s]
        row[trait] = round(sum(vals) / len(vals), 1) if vals else None
    row["n"] = len(scores_list)
    rows.append(row)

df = pd.DataFrame(rows).set_index("model")
print(df.to_string())

csv_path = SCORES_DIR / "summary.csv"
df.to_csv(csv_path)
print(f"\nSaved: {csv_path}")

                           french  allcaps  playful  poetic  skeptical    n
model                                                                      
base                          0.0      2.6      1.8     8.8        7.6  300
ft_french_allcaps            27.1     18.2      2.0     9.7        6.4  300
ft_french_playful            14.4      2.5      6.4    14.0        6.7  300
ft_poetic_skeptical           0.1      3.0      2.8    16.4        8.2  300
ip_french_allcaps_r1          2.9      3.4      1.9    10.5        7.8  300
ip_french_allcaps_r8192      60.0      2.0      2.5     9.9        6.3  300
ip_french_playful             1.8      2.8      1.4    10.6        7.1  300
ip_poetic_skeptical_r1        0.1      2.6      1.8    12.7        6.6  300
ip_poetic_skeptical_r8192     0.0      1.2     19.3    78.6        5.7  300

Saved: ../results/phase2/scores/summary.csv
